# MongoDB + Python in Google Colab
## Step-by-Step CRUD Lab for Beginners

Yes, **MongoDB can be used from Google Colab**.

In this notebook we connect:

**Google Colab → Python → PyMongo → MongoDB Atlas**

We then practise CRUD:

- **C = Create**
- **R = Read**
- **U = Update**
- **D = Delete**

# 1. What is MongoDB?

MongoDB is a **NoSQL document database**.

Instead of mainly using tables, rows, and columns, MongoDB stores information as **documents**.

Example document:

```python
{
    "name": "Emma",
    "subject": "Python",
    "mark": 85
}
```

MongoDB organises information like this:

**Database → Collection → Document**

# 2. SQLite and MongoDB Comparison

| SQLite / SQL | MongoDB |
|---|---|
| Database | Database |
| Table | Collection |
| Row | Document |
| Column | Field |
| INSERT | `insert_one()` |
| SELECT | `find()` |
| UPDATE | `update_one()` |
| DELETE | `delete_one()` |

# 3. What You Need Before Starting

You need:

1. A Google Colab notebook.
2. A MongoDB Atlas account.
3. A MongoDB Atlas database deployment.
4. A MongoDB **database user** with username and password.
5. Your Colab IP address added to the Atlas **IP Access List**.
6. Your Atlas connection string.

The connection string usually starts with:

```text
mongodb+srv://
```

# 4. Find the Current Google Colab Public IP

MongoDB Atlas only accepts connections from IP addresses that are allowed in the project's IP Access List.

Run the next cell and copy the IP address.

Then go to:

**MongoDB Atlas → Network Access → Add IP Address**

If Colab restarts later, the IP can change.

In [ ]:
import requests

colab_ip = requests.get("https://api.ipify.org", timeout=10).text
print("Current Colab public IP:", colab_ip)

# 5. Install PyMongo

`PyMongo` is the official Python driver used to communicate with MongoDB.

In [ ]:
!pip -q install -U pymongo dnspython
print("PyMongo installed.")

# 6. Import MongoDB Tools

`MongoClient` creates the connection between Python and MongoDB.

In [ ]:
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from pprint import pprint
import pandas as pd

print("MongoDB tools are ready.")

# 7. Get Your Atlas Connection String

In MongoDB Atlas:

**Database → Connect → Drivers → Python**

Copy your connection string.

It looks similar to:

```text
mongodb+srv://USERNAME:<db_password>@cluster-name.mongodb.net/?retryWrites=true&w=majority
```

Replace `<db_password>` with your **database user's password**.

We use `getpass()` so the connection string is not displayed on the screen.

In [ ]:
from getpass import getpass

MONGO_URI = getpass("Paste your MongoDB Atlas connection string: ")
print("Connection string received.")

# 8. Create MongoClient

Think of `MongoClient` as the **doorway** between Colab and MongoDB Atlas.

In [ ]:
client = MongoClient(
    MONGO_URI,
    server_api=ServerApi("1"),
    serverSelectionTimeoutMS=10000
)

print("MongoClient created.")

# 9. Test the Connection

We send a small `ping` command.

If MongoDB answers, the connection works.

In [ ]:
try:
    client.admin.command("ping")
    print("SUCCESS: Connected to MongoDB Atlas.")
except Exception as error:
    print("Connection failed.")
    print(error)

# 10. Choose a Database

We will use:

```text
school_database
```

In [ ]:
db = client["school_database"]
print("Database selected:", db.name)

# 11. Choose a Collection

Inside the database we will use:

```text
students
```

A MongoDB **collection** is similar to a SQL table.

In [ ]:
students = db["students"]
print("Collection selected:", students.name)

# 12. Create Our First Python Dictionary

A Python dictionary is very similar to a MongoDB document.

In [ ]:
student = {
    "name": "Emma",
    "subject": "Python",
    "mark": 85,
    "year_level": 5
}

pprint(student)

# CRUD 1 — CREATE

CREATE means **add new data**.

MongoDB uses:

```python
insert_one()
```

to add one document.

In [ ]:
result = students.insert_one(student)

print("Emma added.")
print("Document ID:", result.inserted_id)

# 13. What is `_id`?

MongoDB automatically creates a unique `_id` for each document.

It works like a unique record number.

In [ ]:
pprint(students.find_one({"name": "Emma"}))

# 14. CREATE — Add Several Students

Use:

```python
insert_many()
```

In [ ]:
more_students = [
    {"name": "Noah", "subject": "Database", "mark": 78, "year_level": 5},
    {"name": "Mia", "subject": "Python", "mark": 92, "year_level": 5},
    {"name": "Oliver", "subject": "Cyber Security", "mark": 74, "year_level": 5},
    {"name": "Ava", "subject": "Database", "mark": 88, "year_level": 5}
]

result = students.insert_many(more_students)
print("Students added:", len(result.inserted_ids))

# CRUD 2 — READ

READ means **look at stored data**.

Useful methods:

```python
find_one()
find()
```

# 15. READ — Find One Student

In [ ]:
emma = students.find_one({"name": "Emma"})
pprint(emma)

# 16. READ — Find All Students

In [ ]:
for item in students.find({}):
    pprint(item)

# 17. Show the Documents as a Table

We can convert MongoDB documents to a pandas DataFrame.

In [ ]:
student_list = list(students.find({}))
students_df = pd.DataFrame(student_list)
students_df

# 18. READ — Marks of 80 or Higher

MongoDB uses special comparison operators.

`$gte` means **greater than or equal to**.

In [ ]:
high_marks = list(
    students.find({"mark": {"$gte": 80}})
)

pd.DataFrame(high_marks)

# 19. READ — Only Python Students

In [ ]:
python_students = list(
    students.find({"subject": "Python"})
)

pd.DataFrame(python_students)

# 20. READ — Show Selected Fields Only

The second dictionary in `find()` is called a **projection**.

Here we hide `_id` and show only name, subject, and mark.

In [ ]:
selected = list(
    students.find(
        {},
        {"_id": 0, "name": 1, "subject": 1, "mark": 1}
    )
)

pd.DataFrame(selected)

# CRUD 3 — UPDATE

UPDATE means **change existing data**.

We use:

```python
update_one()
```

and `$set`.

# 21. UPDATE — Change Noah's Mark

In [ ]:
result = students.update_one(
    {"name": "Noah"},
    {"$set": {"mark": 82}}
)

print("Matched:", result.matched_count)
print("Changed:", result.modified_count)

# 22. Check Noah

In [ ]:
pprint(students.find_one({"name": "Noah"}))

# 23. UPDATE — Change More Than One Field

Let's change Oliver's subject and mark.

In [ ]:
students.update_one(
    {"name": "Oliver"},
    {"$set": {"subject": "Python", "mark": 80}}
)

pprint(students.find_one({"name": "Oliver"}))

# 24. UPDATE — Increase a Number

`$inc` means **increase**.

Let's add 2 bonus marks to Mia.

In [ ]:
students.update_one(
    {"name": "Mia"},
    {"$inc": {"mark": 2}}
)

pprint(students.find_one({"name": "Mia"}))

# CRUD 4 — DELETE

DELETE means **remove data**.

We use:

```python
delete_one()
```

# 25. DELETE — Remove Ava

In [ ]:
result = students.delete_one({"name": "Ava"})
print("Deleted:", result.deleted_count)

# 26. Check All Students After DELETE

In [ ]:
pd.DataFrame(
    list(students.find({}))
)

# 27. CRUD Summary

| CRUD | Meaning | MongoDB Method |
|---|---|---|
| Create | Add one | `insert_one()` |
| Create | Add many | `insert_many()` |
| Read | Find one | `find_one()` |
| Read | Find many | `find()` |
| Update | Change one | `update_one()` |
| Update | Change many | `update_many()` |
| Delete | Remove one | `delete_one()` |
| Delete | Remove many | `delete_many()` |

# 28. Count All Students

In [ ]:
total = students.count_documents({})
print("Total students:", total)

# 29. Count Python Students

In [ ]:
python_count = students.count_documents({"subject": "Python"})
print("Python students:", python_count)

# 30. Sort Students by Mark

In [ ]:
sorted_students = list(
    students.find({}, {"_id": 0}).sort("mark", -1)
)

pd.DataFrame(sorted_students)

# 31. Calculate Average Mark With pandas

In [ ]:
df = pd.DataFrame(
    list(students.find({}, {"_id": 0}))
)

print("Average mark:", round(df["mark"].mean(), 2))

# 32. MongoDB Aggregation

MongoDB can also calculate the average inside the database.

`$group` groups data.

`$avg` calculates an average.

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": "$subject",
            "average_mark": {"$avg": "$mark"}
        }
    },
    {
        "$sort": {"average_mark": -1}
    }
]

average_by_subject = list(students.aggregate(pipeline))
pd.DataFrame(average_by_subject)

# 33. MongoDB Can Store Lists

Unlike a simple school table, a document can easily contain a list.

In [ ]:
liam = {
    "name": "Liam",
    "subject": "Python",
    "mark": 90,
    "year_level": 5,
    "hobbies": ["football", "drawing", "coding"]
}

students.insert_one(liam)
pprint(students.find_one({"name": "Liam"}))

# 34. MongoDB Can Store Nested Documents

In [ ]:
sophie = {
    "name": "Sophie",
    "subject": "Database",
    "mark": 86,
    "year_level": 5,
    "contact": {
        "city": "Melbourne",
        "country": "Australia"
    }
}

students.insert_one(sophie)
pprint(students.find_one({"name": "Sophie"}))

# 35. Search Inside Nested Data

MongoDB uses **dot notation**.

Example:

```text
contact.country
```

In [ ]:
australian_students = list(
    students.find(
        {"contact.country": "Australia"},
        {"_id": 0}
    )
)

pd.DataFrame(australian_students)

# 36. Student CRUD Practice

### Task 1 — CREATE
Add Jack:
- subject = Python
- mark = 84
- year_level = 5

### Task 2 — READ
Find Jack.

### Task 3 — UPDATE
Change Jack's mark to 89.

### Task 4 — DELETE
Delete Jack.

### Task 5 — READ
Show all remaining students.

In [ ]:
# Task 1: Write your code here

In [ ]:
# Task 2: Write your code here

In [ ]:
# Task 3: Write your code here

In [ ]:
# Task 4: Write your code here

In [ ]:
# Task 5: Write your code here

# 37. Practice Solutions

In [ ]:
# Solution 1 — CREATE
students.insert_one({
    "name": "Jack",
    "subject": "Python",
    "mark": 84,
    "year_level": 5
})

In [ ]:
# Solution 2 — READ
pprint(students.find_one({"name": "Jack"}))

In [ ]:
# Solution 3 — UPDATE
students.update_one(
    {"name": "Jack"},
    {"$set": {"mark": 89}}
)

pprint(students.find_one({"name": "Jack"}))

In [ ]:
# Solution 4 — DELETE
students.delete_one({"name": "Jack"})
print("Jack deleted.")

In [ ]:
# Solution 5 — READ ALL
pd.DataFrame(list(students.find({}, {"_id": 0})))

# 38. Export MongoDB Data to CSV

In [ ]:
export_df = pd.DataFrame(
    list(students.find({}, {"_id": 0}))
)

export_df.to_csv("mongodb_students.csv", index=False)
print("mongodb_students.csv created.")

# 39. Download the CSV From Google Colab

In [ ]:
from google.colab import files
files.download("mongodb_students.csv")

# 40. Important Safety Rules

1. Never publish your real MongoDB password.
2. Never place a real connection string in a public notebook.
3. Use a database user with only the permissions needed.
4. Keep the Atlas IP Access List restricted.
5. Remove temporary access when the class is finished.
6. Use classroom sample data, not private personal data.

# 41. Close the MongoDB Connection

In [ ]:
client.close()
print("MongoDB connection closed.")

# Final Review

You have connected **Google Colab to MongoDB Atlas** using Python.

### Main picture

```text
Google Colab
    ↓
Python
    ↓
PyMongo / MongoClient
    ↓
Internet
    ↓
MongoDB Atlas
    ↓
Database
    ↓
Collection
    ↓
Documents
```

### CRUD

**CREATE**
```python
insert_one()
insert_many()
```

**READ**
```python
find_one()
find()
```

**UPDATE**
```python
update_one()
update_many()
```

**DELETE**
```python
delete_one()
delete_many()
```